# 05 — Pre-dam Gauge Site Map (Exploratory)

Two-layer interactive map:
- **Layer 1 — Gauge sites**: stream discharge stations from restored USGS metadata, color-coded by the earliest decade of their measurement record. Pre-1940 layers shown by default.
- **Layer 2 — NID Dams**: all California dams from the National Inventory of Dams (downloaded May 2026).

Goal: visually assess which rivers were gauged before major impoundment as a precursor to a formal pre-dam baseline analysis.

Requires:
- `data/analysis/processed_metadata.parquet` (output of `00_data_prep.ipynb`)
- `data/analysis/dams.csv` (NID California export)

Output: `manuscript/figures/predam_gauge_map.html`

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

root         = Path('../../..')
parquet_path = root / 'data/analysis/processed_metadata.parquet'
nid_path     = root / 'data/analysis/dams.csv'
figures_path = root / 'manuscript/figures'
figures_path.mkdir(parents=True, exist_ok=True)

df   = pd.read_parquet(parquet_path)
dams = pd.read_csv(nid_path)

print(f'Metadata:  {len(df):,} rows')
print(f'NID dams:  {len(dams):,} rows')

## Filter and deduplicate gauge sites

Keep stream discharge rows that have parsed coordinates within California's bounding box and a parsed `year_start`. Round coordinates to 2 decimal places (~1 km) and group by `(lat, lon, watersource_name)` to collapse multiple documents referencing the same physical station into one site record.

In [2]:
CA_LAT = (32.5, 42.1)
CA_LON = (-124.6, -114.1)

sd = df[
    (df['water_type_clean'] == 'Stream Discharge') &
    df['lat_combined'].notna() &
    df['lon_combined'].notna() &
    df['year_start'].notna() &
    df['lat_combined'].between(*CA_LAT) &
    df['lon_combined'].between(*CA_LON)
].copy()

print(f'Stream discharge rows (CA, with coords + year_start): {len(sd):,}')

sd['lat_r'] = sd['lat_combined'].round(2)
sd['lon_r'] = sd['lon_combined'].round(2)
sd['source_norm'] = sd['watersource_name'].fillna('').str.lower().str.strip()

sites = (
    sd.groupby(['lat_r', 'lon_r', 'source_norm'], as_index=False)
    .agg(
        watersource_name   = ('watersource_name',    'first'),
        year_start         = ('year_start',           'min'),
        year_end           = ('year_end',             'max'),
        coord_source       = ('coord_source',         'first'),
        n_docs             = ('id',                   'nunique'),
        temporal_res       = ('temporal_resolution',  lambda x: ', '.join(sorted(set(x.dropna()))[:3])),
    )
)

print(f'Unique gauge sites after deduplication: {len(sites):,}')
print(f'  coord_source breakdown:')
print(sites['coord_source'].value_counts().to_string())

Stream discharge rows (CA, with coords + year_start): 57,577


Unique gauge sites after deduplication: 34,692
  coord_source breakdown:
coord_source
actual      19003
inferred    15689


## Assign measurement era

Era is based on `year_start` — the earliest year a measurement record at this site appears in the dataset. This is the year the site was *first gauged* in our restored records, not necessarily the first year the station ever operated.

In [3]:
ERA_ORDER  = ['Pre-1900', '1900–1919', '1920–1939', '1940–1959', '1960+']
ERA_COLORS = {
    'Pre-1900':  '#1d3d8f',   # dark navy
    '1900–1919': '#2166ac',   # medium blue
    '1920–1939': '#27a86e',   # teal-green
    '1940–1959': '#e8912b',   # amber
    '1960+':     '#c0392b',   # red
}

def assign_era(y):
    if y < 1900: return 'Pre-1900'
    if y < 1920: return '1900–1919'
    if y < 1940: return '1920–1939'
    if y < 1960: return '1940–1959'
    return '1960+'

sites['era'] = sites['year_start'].apply(assign_era)

summary = sites.groupby('era').agg(n_sites=('lat_r', 'count')).reindex(ERA_ORDER)
summary['pct'] = (summary['n_sites'] / len(sites) * 100).round(1)
print('Sites by era:')
print(summary.to_string())

Sites by era:
           n_sites   pct
era                     
Pre-1900      1025   3.0
1900–1919     9078  26.2
1920–1939     5941  17.1
1940–1959     4310  12.4
1960+        14338  41.3


## Prepare NID dams

Filter to dams within California's bounding box. Dams without a year completed are retained but labelled 'Unknown'.

In [4]:
dams['Year Completed'] = pd.to_numeric(dams['Year Completed'], errors='coerce')

dams_ca = dams[
    dams['Latitude'].notna() &
    dams['Longitude'].notna() &
    pd.to_numeric(dams['Latitude'],  errors='coerce').between(*CA_LAT) &
    pd.to_numeric(dams['Longitude'], errors='coerce').between(*CA_LON)
].copy()

dams_ca['Latitude']  = dams_ca['Latitude'].astype(float)
dams_ca['Longitude'] = dams_ca['Longitude'].astype(float)

print(f'NID dams in CA bounding box: {len(dams_ca):,}')
print(f'  With year completed: {dams_ca["Year Completed"].notna().sum():,}')
print(f'  Without year: {dams_ca["Year Completed"].isna().sum():,}')
print(f'  Year range: {int(dams_ca["Year Completed"].min())} – {int(dams_ca["Year Completed"].max())}')

NID dams in CA bounding box: 1,534
  With year completed: 1,456
  Without year: 78
  Year range: 1850 – 2022


## Build interactive map

Built with Plotly (WebGL rendering) so it handles 34,000+ points without browser lag.

**Layer 1 — Gauge sites**: one trace per era, color-coded. Pre-1940 eras are visible by default; 1940+ are in the legend but hidden — click to toggle.

**Layer 2 — NID Dams**: all CA dams as dark star markers. Hover for name, river, year completed, and storage.

Saved as a self-contained HTML file to `manuscript/figures/`.

In [5]:
# ── Prep: string-safe columns for hover text ──────────────────────────────────
sites['yr_start_str'] = sites['year_start'].astype(int).astype(str)
sites['yr_end_str']   = sites['year_end'].apply(lambda x: str(int(x)) if pd.notna(x) else '?')

dams_ca['yr_str']      = dams_ca['Year Completed'].apply(
    lambda x: str(int(x)) if pd.notna(x) else 'Unknown'
)
dams_ca['storage_str'] = dams_ca['NID Storage (Acre-Ft)'].apply(
    lambda x: f'{x:,.0f}' if pd.notna(x) and x > 0 else '—'
)

# ── Build figure ───────────────────────────────────────────────────────────────
fig = go.Figure()

# Layer 1: gauge sites, one trace per era
for era in ERA_ORDER:
    sub     = sites[sites['era'] == era]
    visible = True if era in ('Pre-1900', '1900–1919', '1920–1939') else 'legendonly'

    fig.add_trace(go.Scattermap(
        lat      = sub['lat_r'],
        lon      = sub['lon_r'],
        mode     = 'markers',
        marker   = dict(size=6, color=ERA_COLORS[era], opacity=0.75),
        name     = f'Gauges: {era}  (n={len(sub):,})',
        visible  = visible,
        hovertemplate = (
            '<b>%{customdata[0]}</b><br>'
            'Record: %{customdata[1]}–%{customdata[2]}<br>'
            'Coord source: %{customdata[3]}<br>'
            'Documents: %{customdata[4]}<br>'
            'Resolution: %{customdata[5]}'
            '<extra></extra>'
        ),
        customdata = sub[['watersource_name', 'yr_start_str', 'yr_end_str',
                           'coord_source', 'n_docs', 'temporal_res']].fillna('—').values,
    ))

# Layer 2: NID dams
fig.add_trace(go.Scattermap(
    lat      = dams_ca['Latitude'],
    lon      = dams_ca['Longitude'],
    mode     = 'markers',
    marker   = dict(size=9, color='#1a1a1a', opacity=0.85, symbol='star'),
    name     = f'NID Dams  (n={len(dams_ca):,})',
    visible  = True,
    hovertemplate = (
        '<b>%{customdata[0]}</b><br>'
        'River: %{customdata[1]}<br>'
        'Year completed: %{customdata[2]}<br>'
        'Storage: %{customdata[3]} acre-ft<br>'
        'Status: %{customdata[4]}<br>'
        'Hazard: %{customdata[5]}'
        '<extra></extra>'
    ),
    customdata = dams_ca[['Dam Name', 'River or Stream Name', 'yr_str', 'storage_str',
                           'Operational Status', 'Hazard Potential Classification']].fillna('—').values,
))

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    map=dict(
        style='carto-positron',
        center=dict(lat=37.5, lon=-119.5),
        zoom=5,
    ),
    title=dict(
        text='California stream gauge sites vs. NID dams — exploratory pre-dam inventory',
        x=0.5, xanchor='center', font=dict(size=14),
    ),
    legend=dict(
        yanchor='top', y=0.99, xanchor='left', x=0.01,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='rgba(0,0,0,0.15)', borderwidth=1,
        font=dict(size=11),
    ),
    height=720,
    margin=dict(l=0, r=0, t=45, b=0),
)

fig.show()

In [6]:
out_path = figures_path / 'predam_gauge_map.html'
fig.write_html(str(out_path), include_plotlyjs='cdn')
print(f'Saved: {out_path}')

Saved: ..\..\manuscript\figures\predam_gauge_map.html
